# 05 Slices, fairness and SHAP: who does the model let down, and why?

Day 2, 14:10 to 14:45 (Lab 4). Lab sheet: `Day2/labs/lab4-slices-fairness-shap.md`.

An overall recall of 0.84 can hide a product type where the model misses far more. Today we check
recall and missed failures for product types **L, M and H** with Fairlearn, then use SHAP to see
which sensor readings drove a prediction. You write up what you find in `reports/slice_explainability_note.md`.

In [ ]:
import os
import sys
from pathlib import Path

# Work from the repo root (the folder that holds machine_guard/), wherever Jupyter started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "machine_guard").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working in", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from machine_guard.features import REGISTERED_MODEL_NAME, RAW_FEATURES, TARGET, build_pipeline, load_data, split
from machine_guard.train import setup_mlflow

df = load_data()
X_train, X_test, y_train, y_test = split(df)

setup_mlflow()
try:
    model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}@champion")
    source = "registry: machine-guard@champion"
except Exception as err:  # no registry yet: rebuild the same champion (same settings, same seed)
    print("Could not load from the registry, rebuilding the champion instead:", err)
    model = build_pipeline().fit(X_train, y_train)
    source = "rebuilt with build_pipeline()"

pred = model.predict(X_test)
print("Model from", source)
print(f"Overall: caught {int(((y_test == 1) & (pred == 1)).sum())} of {int(y_test.sum())} failures")

## 1. Slice by product type with Fairlearn's `MetricFrame`

`MetricFrame` computes the same metrics once overall and once per group. The group column here is
`type` (L = low, M = medium, H = high quality variant). For people, it would be age, sex or ethnicity.

In [ ]:
from fairlearn.metrics import MetricFrame, count
from sklearn.metrics import precision_score, recall_score


def failures(y_true, y_pred):
    return int(y_true.sum())


def missed(y_true, y_pred):
    return int(((y_true == 1) & (y_pred == 0)).sum())


def false_alarms(y_true, y_pred):
    return int(((y_true == 0) & (y_pred == 1)).sum())


def precision(y_true, y_pred):
    return precision_score(y_true, y_pred, zero_division=0)


# TODO: build the MetricFrame. Give it
#   metrics={"runs": count, "failures": failures, "recall": recall_score, "missed": missed,
#            "false_alarms": false_alarms, "precision": precision},
#   y_true=y_test, y_pred=pred and sensitive_features=X_test["type"]
by_type = ...

print("Overall:", by_type.overall.round(3).to_dict())
by_type.by_group.loc[["L", "M", "H"]].round(3)

In [ ]:
print("Largest recall gap between types:", round(by_type.difference()["recall"], 3))
print("Worst / best recall ratio:       ", round(by_type.ratio()["recall"], 3))

ax = by_type.by_group.loc[["L", "M", "H"], "recall"].plot.bar(color=["#1f3a5f", "#c98a00", "#2e7d4f"], rot=0)
ax.axhline(by_type.overall["recall"], color="grey", linestyle="--", label="overall")
for i, (r, n) in enumerate(by_type.by_group.loc[["L", "M", "H"], ["recall", "failures"]].values):
    ax.text(i, r + 0.02, f"{r:.2f}\n({int(n)} failures)", ha="center")
ax.set_ylim(0, 1.15)
ax.set_ylabel("recall (share of failures caught)")
ax.set_title("Champion recall by product type, test set")
ax.legend()
plt.show()

**Stop and think.** How many failures does type H have in the test set? If the model missed one more,
how much would H's recall drop? Can you trust a slice number built on so few cases?

## 2. Is the gap real? Use all 10,000 runs

The test set has only a handful of H failures. Cross-validation trains 5 models, each tested on a
different fifth of the data, so every run gets one honest prediction. That gives about five times as many
failures per slice. (It retrains the champion's settings 5 times: under a minute.)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X_all, y_all = df[RAW_FEATURES], df[TARGET]
cv_pred = cross_val_predict(build_pipeline(), X_all, y_all, cv=StratifiedKFold(5, shuffle=True, random_state=42))

by_type_cv = MetricFrame(
    metrics={"failures": failures, "recall": recall_score, "missed": missed, "false_alarms": false_alarms},
    y_true=y_all, y_pred=cv_pred, sensitive_features=df["type"],
)
print("Overall:", by_type_cv.overall.round(3).to_dict())
by_type_cv.by_group.loc[["L", "M", "H"]].round(3)

## 3. Which failures does it miss?

The failure-mode columns (`twf` to `osf`) are never model inputs (leakage), but we can use them
**after** predicting to understand the mistakes.

In [ ]:
modes = ["twf", "hdf", "pwf", "osf", "rnf"]
test_rows = df.loc[X_test.index]
caught_rows = test_rows[(y_test == 1) & (pred == 1)]
missed_rows = test_rows[(y_test == 1) & (pred == 0)]

summary = pd.DataFrame({"caught": caught_rows[modes].sum(), "missed": missed_rows[modes].sum()})
summary.index = ["tool wear (TWF)", "heat dissipation (HDF)", "power (PWF)", "overstrain (OSF)", "random (RNF)"]
print("Missed failures with no mode recorded:", int((missed_rows[modes].sum(axis=1) == 0).sum()))
summary

In [ ]:
missed_rows[["type", "tool_wear_min", "torque_nm", "rotational_speed_rpm", "twf"]].sort_values("tool_wear_min")

## 4. SHAP: which readings push the risk up or down?

SHAP splits one prediction into contributions, one per input. Positive values push towards "fail",
negative towards "OK". They add up from the model's average output to this run's predicted probability.

The model inside the pipeline sees scaled numbers. To keep the plots readable we show the readings in
their real units (kelvin, rpm, Nm, minutes) next to the SHAP values.

In [ ]:
import shap

from machine_guard.features import add_features

prepare = model[:-1]                      # everything before the forest: features, encoding, scaling
forest = model.named_steps["model"]
X_model = prepare.transform(X_test)       # what the forest actually sees

explainer = shap.TreeExplainer(forest)
values = explainer.shap_values(X_model)
values = values[:, :, 1] if np.ndim(values) == 3 else values[1]   # contributions to the "fail" class

names = [n.split("__")[1] for n in model.named_steps["prepare"].get_feature_names_out()]
readable = add_features(X_test)
readable = pd.concat([pd.get_dummies(readable["type"], prefix="type").reindex(columns=["type_L", "type_M", "type_H"])
                      .astype(int), readable[names[3:]].round(1)], axis=1)
explanation = shap.Explanation(values=values, base_values=np.full(len(values), explainer.expected_value[1]),
                               data=readable.values, feature_names=names)
shap.plots.bar(explanation, show=False)
plt.title("Average impact on the failure probability (test set)")
plt.show()

In [ ]:
shap.plots.beeswarm(explanation, show=False)
plt.title("Each dot is one test run: red = high reading, blue = low")
plt.show()

## 5. One run at a time

A waterfall plot explains a single prediction: start at the average risk at the bottom, add each
reading's push, and arrive at this run's probability at the top. Use it the way a technician would:
"why did the alarm go off on this machine?"

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
pos = np.arange(len(X_test))
caught_pos = pos[(y_test.values == 1) & (pred == 1)]
missed_pos = pos[(y_test.values == 1) & (pred == 0)]

alarm = caught_pos[np.argmax(proba[caught_pos])]      # the alarm the model was most sure about
print(f"Caught failure, row {X_test.index[alarm]}: probability {proba[alarm]:.2f}")
shap.plots.waterfall(explanation[alarm], show=False)
plt.show()

In [ ]:
# TODO: explain one MISSED failure. Pick a position from missed_pos (for example missed_pos[0])
#   and draw its waterfall the same way as the cell above.
print("Missed failures at positions:", missed_pos)
miss = ...

## 6. Write it up

Open `reports/slice_explainability_note.md` and fill it in: one page, plain words, numbers from this
notebook. Then commit both files:

```bash
git add notebooks/05_slices_fairness_shap.ipynb reports/slice_explainability_note.md
git commit -m "Slice, fairness and SHAP analysis of the champion"
git push
```